In [1]:
# dependencies

import pandas as pd 
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [3]:
# data 

df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
# age , fever => num col
# gender, city, => nominal categorical
# cough => odinal categorical

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [9]:
# as we can see that tere are 100 rows but there are only 90 non null in fever - so 10 missing values

In [10]:
# or we can also do this 

df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [11]:
df.columns

Index(['age', 'gender', 'fever', 'cough', 'city', 'has_covid'], dtype='object')

In [12]:
# so the preprocessing
# geneder - ohe
# fever - impuder
# cough - odinal encoder
# city - ohe 
# has_covid - label encoder


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
x_train,x_test, y_train,y_test = train_test_split(df.drop(columns=['has_covid']),
                                                 df['has_covid'],
                                                 test_size=0.2,
                                                 random_state=1)

In [17]:
x_train.shape,x_test.shape, y_train.shape,y_test.shape

((80, 5), (20, 5), (80,), (20,))

# 1. Aam Zindagi

In [19]:
# doing every step manually

# fever - impuder

# make object of simplesimputer class
si = SimpleImputer()

# fit on train and trasform both

x_train_fever = si.fit_transform(x_train[['fever']])

In [20]:
x_test_fever = si.transform(x_test[['fever']])

In [21]:
# missing values are replaced by data mean

x_test_fever.shape

(20, 1)

In [24]:
# odinal encoding-->cough

oe = OrdinalEncoder(categories=[['Mild','Strong']])

x_train_cough = oe.fit_transform(x_train[['cough']])

x_test_cough = oe.transform(x_test[['cough']])

In [26]:
x_test_cough.shape


(20, 1)

In [32]:
# one hot encoding --> city, gender
# here we have 4 value in city and 2 in gender -- when we drop first col
# final result = 3 col of city_value and 1 of Gender_value

ohe = OneHotEncoder(drop='first', sparse_output=False)
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])

# also for test 

x_test_gender_city = ohe.transform(x_test[['gender','city']])

In [33]:
x_test_gender_city
# here in array first value is of gender and rest 3 are of city

array([[0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [1., 1., 0., 0.],
       [1., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [1., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [1., 0., 1., 0.],
       [1., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.]])

In [36]:
# now we will concatinate all , for this we also need to extract age of data

x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

In [38]:
x_test_age.shape

(20, 1)

In [41]:
# now we will concatinate

x_train_transformed = np.concatenate((x_train_age,x_train_fever,x_train_gender_city,x_train_cough),axis=1)

x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)

In [42]:
x_train_transformed.shape

(80, 7)

# 2. Mintoss zindagi

In [44]:
df.head()


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [43]:
from sklearn.compose import ColumnTransformer

In [51]:
ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(), ['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city']),
] ,
                               remainder='passthrough')

ColumnTransformer(remainder='passthrough',
                  transformers=[('tnf1', SimpleImputer(), ['fever']),
                                ('tnf2',
                                 OrdinalEncoder(categories=[['Mild',
                                                             'Strong']]),
                                 ['cough']),
                                ('tnf3',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city'])])

In [45]:
# remainder -- drop / passthrough
# transformers = passed in list of tupples [(var_name, Transformation type, ['col_name'])]

transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(), ['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city']),
] ,
                               remainder='passthrough')

In [46]:
transformer.fit_transform(x_train)

array([[101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [ 98.        ,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  34.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  20.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  56.        ],
       [100.95833333,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  42.        ],
       [102.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  82.        ],
       [100.95833333,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  79.        ],
       [102.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  33.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.    

In [47]:
transformer.fit_transform(x_train).shape

(80, 7)

In [48]:
transformer.fit_transform(x_test)

array([[ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  69.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  26.        ],
       [ 99.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  65.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  27.        ],
       [ 98.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  40.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  38.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  24.        ],
       [103.        ,   0.        ,   0.        ,   1.        ,
          0.    

In [49]:
transformer.fit_transform(x_test).shape

(20, 7)

In [50]:
# done